In [0]:
%pip install -U \
langchain \
langchain-community \
langchain-core \
langchain-databricks \
faiss-cpu \
pypdf

INFO: pip is looking at multiple versions of langchain-databricks to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-databricks to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to r

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_databricks import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct"
)

/home/spark-45b1e7d9-0c42-47a7-8f72-8d/.ipykernel/305/command-6380370399602472-2710930687:3: LangChainDeprecationWarning: Use databricks_langchain.ChatDatabricks
  llm = ChatDatabricks(


In [0]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
Databricks is a unified analytics platform built on Apache Spark.
It provides Delta Lake, MLflow, Unity Catalog and AI capabilities.
"""
    ),
    Document(
        page_content="""
LangChain helps developers build applications by combining LLMs,
retrievers, tools and prompts.
"""
    ),
]

In [0]:
# Use Databricks embeddings.
from langchain_databricks import DatabricksEmbeddings

embedding = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

/home/spark-45b1e7d9-0c42-47a7-8f72-8d/.ipykernel/305/command-6380370399602474-2228005517:4: LangChainDeprecationWarning: Use databricks_langchain.DatabricksEmbeddings
  embedding = DatabricksEmbeddings(


In [0]:
# Create FAISS

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents,
    embedding
)



In [0]:
# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k":2}
)


In [0]:
from langchain_core.prompts import ChatPromptTemplate

qa_prompt = ChatPromptTemplate.from_messages(
[
(
"system",
"""
Answer using only the supplied context.

If you don't know the answer,
say you don't know.

Context:

{context}
"""
),
("human","{input}")
]
)

In [0]:
 # Retrieval Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

qa_chain = create_retrieval_chain(
    retriever,
    document_chain
)


In [0]:
# testing 
result = qa_chain.invoke(
{
"input":"What is Databricks?"
}
)

print(result["answer"])

Databricks is a unified analytics platform built on Apache Spark. It provides Delta Lake, MLflow, Unity Catalog, and AI capabilities.


In [0]:
# second chain

from langchain_core.output_parsers import StrOutputParser

social_prompt = ChatPromptTemplate.from_template(
"""
Create a LinkedIn post from this summary.

Summary:

{summary}
"""
)

In [0]:
# multi Stage chain

multi_chain = (
    {"summary": qa_chain}
    | social_prompt
    | llm
    | StrOutputParser()
)

In [0]:
# call

result = multi_chain.invoke(
{
"input":"What is LangChain?"
}
)

print(result)

Here's a potential LinkedIn post based on the summary:

**Unlock the Power of AI-Driven Development with LangChain!**

Are you looking to build innovative applications that leverage the latest advancements in artificial intelligence? Look no further than LangChain!

LangChain is a game-changing platform that enables developers to combine Large Language Models (LLMs), retrievers, tools, and prompts to create powerful applications. By harnessing the potential of these technologies, you can unlock new possibilities for your business and stay ahead of the curve.

Whether you're working with data analytics platforms like Databricks, or exploring new use cases for AI, LangChain provides the foundation you need to succeed. With its flexible and intuitive architecture, you can focus on building applications that drive real value for your organization.

Join the LangChain community today and discover the endless possibilities of AI-driven development! #LangChain #AI #LLMs #ApplicationDevelopmen

In [0]:
# add thrid stage

quiz_prompt = ChatPromptTemplate.from_template(
"""
Create five interview questions from this text.

{text}
"""
)

quiz_chain = (
    quiz_prompt
    | llm
    | StrOutputParser()
)

pipeline = (
    {"summary": qa_chain}
    | social_prompt
    | llm
    | StrOutputParser()
    | quiz_chain
)

In [0]:
result = pipeline.invoke(
{
"input":"What is LangChain?"
}
)

print(result)

Here are five interview questions based on the text:

1. Can you explain how LangChain enables developers to leverage Large Language Models (LLMs) in their application development, and what benefits does this bring to the development process?

2. How do you think AI-driven development, as facilitated by LangChain, will change the future of software development, and what opportunities or challenges do you see arising from this shift?

3. What experience do you have with data analytics platforms like Databricks, and how do you think LangChain can be used to enhance or integrate with these platforms?

4. How do you stay up-to-date with the latest advancements in artificial intelligence, and what role do you think LangChain can play in helping developers like yourself to harness the power of AI in their work?

5. Can you describe a potential use case for LangChain that you think would be particularly innovative or impactful, and how you would approach building an application using the Lang